# Clase 2 — Errores comunes y patrones de prompting

Antes de aprender técnicas avanzadas, vale la pena aprender a **diagnosticar** por qué un prompt no funciona. La mayoría de los problemas con los modelos de lenguaje se reducen a cinco errores recurrentes — y todos tienen solución.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Los cinco anti-patrones más frecuentes |
| 3 | Herramienta de diagnóstico |
| 4 | Reformulación sistemática |
| 5 | Actividad: corregir prompts rotos |

---
## 1. Configuración del entorno

Este notebook usa el mismo wrapper de la clase anterior. Si ya configuraste tu entorno, solo cambiá `BACKEND` y continuá.

**Cómo guardar tu API key de Gemini (si es la primera vez):**
1. Obtené tu key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. En tu terminal, desde la carpeta del proyecto:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. El archivo `.env` queda en tu máquina y nunca se sube al repositorio.

Si preferís no crear el archivo, la celda siguiente te pide la clave de forma interactiva.

In [1]:
import os
import getpass

BACKEND = "ollama"          # "gemini", "ollama", "local"
GEMINI_MODEL = "gemma-4-26b-a4b-it"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini (no se muestra): ")

print(f"Backend: {BACKEND}")

Backend: ollama


In [2]:
# ─── Inicializar cliente y wrapper ────────────────────────────────────────────

if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

elif BACKEND == "ollama":
    import ollama
    
    OLLAMA_MODEL = "gemma2:9b"  # Modelo que tenés descargado
    
    print("🚀 Conectando a Ollama...")
    try:
        ollama.list()
        print(f"✅ Ollama disponible. Usando modelo: {OLLAMA_MODEL}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Solución: Abre otra terminal y ejecuta: ollama serve")
        raise

elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    print("Descargando modelo local...")

    ruta_modelo = hf_hub_download(
        repo_id="unsloth/gemma-3-1b-it-GGUF",
        filename="gemma-3-1b-it-Q4_K_M.gguf"
    )
    print("Inicializando modelo local...")
    _llm_local = Llama(
        model_path=ruta_modelo,
        n_ctx=2048,
        n_gpu_layers=0,
        verbose=False
    )
    print("✅ Modelo local listo (usando CPU).")


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=200):
    
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    
    elif BACKEND == "ollama":
        r = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            system=system_prompt,
            stream=False,
            options={
                "temperature": temperature,
                "num_predict": max_tokens,
            }
        )
        return r['response'].strip()

    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

🚀 Conectando a Ollama...
✅ Ollama disponible. Usando modelo: gemma2:9b
Entorno listo.


---
## 2. Los cinco anti-patrones más frecuentes

Un **anti-patrón** es una forma de escribir prompts que parece razonable pero produce resultados pobres de forma predecible.

| # | Anti-patrón | Síntoma | Causa raíz |
|---|---|---|---|
| 1 | **Instrucción vaga** | Respuesta genérica o demasiado larga | No se especificó la tarea ni el alcance |
| 2 | **Sin rol** | Tono y nivel inapropiados para el destinatario | El modelo elige el perfil por defecto |
| 3 | **Sin formato** | Texto en prosa cuando necesitás lista, o viceversa | El modelo estructura a su criterio |
| 4 | **Múltiples tareas en un prompt** | Solo responde la primera, ignora el resto | Ambigüedad sobre prioridad |
| 5 | **Negaciones sin alternativa** | Cumple la negación de forma inesperada | El modelo se enfoca en lo que *no* debe hacer |

> 💡 Los anti-patrones 1, 2 y 3 son los más comunes entre quienes recién empiezan. El 4 y el 5 aparecen cuando ya se tiene algo de experiencia pero no se estructura bien la complejidad.

---
## 3. Herramienta de diagnóstico

Antes de reescribir un prompt que no funcionó, conviene hacerse cuatro preguntas rápidas:

In [3]:
def diagnosticar_prompt(prompt):
    """Imprime un diagnóstico rápido del prompt ingresado."""

    print("─" * 55)
    print("DIAGNÓSTICO DEL PROMPT")
    print("─" * 55)
    print(f"Texto: {prompt[:80]}{'...' if len(prompt) > 80 else ''}")
    print()

    # Indicadores simples basados en el texto
    tiene_rol     = any(p in prompt.lower() for p in ["sos ", "eres ", "actuá ", "actúa "])
    tiene_formato = any(p in prompt.lower() for p in ["lista", "tabla", "puntos", "bullets", "formato", "columnas"])
    es_corto      = len(prompt.split()) < 8
    tiene_no      = prompt.lower().startswith("no ") or " no " in prompt.lower()[:30]
    tiene_y_ademas = prompt.count(".") > 2 or " y además" in prompt.lower() or " también" in prompt.lower()

    checks = [
        ("Tiene rol explícito",        tiene_rol,         "Agregá 'Sos un...' al comienzo"),
        ("Especifica el formato",       tiene_formato,     "Indicá cómo querés la respuesta: lista, tabla, párrafo..."),
        ("Instrucción suficientemente larga", not es_corto, "La instrucción parece muy corta; agregá contexto o alcance"),
        ("Sin negaciones como punto de partida", not tiene_no, "Reescribí usando lo que SÍ querés en vez de lo que no querés"),
        ("Una tarea a la vez",           not tiene_y_ademas, "Separar en dos prompts si hay múltiples tareas"),
    ]

    for nombre, ok, consejo in checks:
        estado = "✓" if ok else "✗"
        print(f"  {estado}  {nombre}")
        if not ok:
            print(f"       → {consejo}")

    print("─" * 55)


# Probamos con un prompt claramente deficiente
diagnosticar_prompt("Habla de abogacía.")

───────────────────────────────────────────────────────
DIAGNÓSTICO DEL PROMPT
───────────────────────────────────────────────────────
Texto: Habla de abogacía.

  ✗  Tiene rol explícito
       → Agregá 'Sos un...' al comienzo
  ✗  Especifica el formato
       → Indicá cómo querés la respuesta: lista, tabla, párrafo...
  ✗  Instrucción suficientemente larga
       → La instrucción parece muy corta; agregá contexto o alcance
  ✓  Sin negaciones como punto de partida
  ✓  Una tarea a la vez
───────────────────────────────────────────────────────


In [4]:
# Ahora con un prompt mejor estructurado
diagnosticar_prompt(
    "Sos un especialista en marketing digital. "
    "Listá 4 estrategias para aumentar seguidores en Instagram para una marca de ropa sustentable."
)

───────────────────────────────────────────────────────
DIAGNÓSTICO DEL PROMPT
───────────────────────────────────────────────────────
Texto: Sos un especialista en marketing digital. Listá 4 estrategias para aumentar segu...

  ✓  Tiene rol explícito
  ✓  Especifica el formato
  ✓  Instrucción suficientemente larga
  ✓  Sin negaciones como punto de partida
  ✓  Una tarea a la vez
───────────────────────────────────────────────────────


---
## 4. Reformulación sistemática

Para cada anti-patrón hay una reformulación directa. Vamos a ver los cinco en acción comparando la versión rota con la versión corregida.

In [5]:
# ─── Anti-patrón 1: instrucción vaga ─────────────────────────────────────────
roto    = "Escribí algo sobre liderazgo."
corregido = "Sos un coach ejecutivo. Escribí 3 principios de liderazgo aplicables a equipos remotos, en formato de lista numerada, máximo 2 líneas por punto."

print("ROTO:\n", "Prompt:",roto)
print(llamar_llm(roto, max_tokens=100))
print("\n" + "----" * 20)
print()
print("CORREGIDO: \n", "Prompt:", corregido[:60], "...")
print(llamar_llm(corregido, max_tokens=120))

ROTO:
 Prompt: Escribí algo sobre liderazgo.
El liderazgo eficaz se basa en la capacidad de inspirar, motivar e influir en otros para alcanzar objetivos comunes. 

**Características clave:**

* **Visón clara:** Definir una dirección y propósito compartido.
* **Comunicación efectiva:** Transmitir ideas con claridad y empatía.
* **Inteligencia emocional:** Comprender y gestionar las emociones propias y ajenas.
* **Toma de decisiones informada:** Evaluar opciones y tomar acciones estratégicas.
*

--------------------------------------------------------------------------------

CORREGIDO: 
 Prompt: Sos un coach ejecutivo. Escribí 3 principios de liderazgo ap ...
Aquí tienes 3 principios de liderazgo aplicables a equipos remotos:

1. **Comunícate claramente y con frecuencia:** Establece canales de comunicación eficientes y utiliza herramientas colaborativas para mantener al equipo informado y conectado. La transparencia y la accesibilidad son clave.
2. **Confía en tu equipo y delega respons

In [6]:
# ─── Anti-patrón 2: sin rol ──────────────────────────────────────────────
roto    = "Escribí un mensaje para rechazar a un candidato después de una entrevista."
corregido = """Sos un reclutador experimentado en una empresa de tecnología, conocido por tu empatía.
Escribí un mensaje de correo electrónico (máximo 3 párrafos) para rechazar a un candidato después de una
entrevista, manteniendo un tono profesional, alentador y humano."""

print("ROTO:")
print(llamar_llm(roto, max_tokens=200))
print("----" * 20)
print()
print("CORREGIDO:")
print(llamar_llm(corregido, max_tokens=200))

ROTO:
Estimado/a [Nombre del candidato],

Gracias por su tiempo e interés en la posición de [nombre de la posición] en [nombre de la empresa]. 

Si bien apreciamos mucho su perfil y experiencia, hemos decidido seguir adelante con otro candidato cuyo perfil se ajusta mejor a las necesidades actuales del puesto. 

Le deseamos mucho éxito en su búsqueda laboral.


Atentamente,
[Tu nombre]
--------------------------------------------------------------------------------

CORREGIDO:
Asunto: Actualización sobre el puesto de [Nombre del puesto] en [Nombre de la empresa]

Estimado/a [Nombre del candidato],

Gracias por tu tiempo e interés en el puesto de [Nombre del puesto] en [Nombre de la empresa]. Tu experiencia en [mencionar alguna habilidad o experiencia relevante] es realmente impresionante. Después de una cuidadosa consideración, hemos decidido seguir adelante con otros candidatos cuyos perfiles se ajustan aún mejor a las necesidades específicas de este rol en este momento.

Te deseo lo 

In [7]:
# ─── Anti-patrón 3: sin formato ──────────────────────────────────────────
roto    = "Decime los ingredientes para hacer milanesas."
corregido = """Listá los ingredientes para hacer milanesas.
Formato requerido: una lista con viñetas (bullet points), donde cada punto indique cantidad y nombre del ingrediente."""

print("ROTO:")
print(llamar_llm(roto, max_tokens=150))
print("----" * 20)
print("CORREGIDO:")
print(llamar_llm(corregido, max_tokens=150))

ROTO:
## Ingredientes para Milanesas:

* **Carne:** 500g de carne (res, cerdo o pollo) en rodajas finas
* **Harina:** 1 taza
* **Huevos:** 2 grandes, batidos
* **Pan rallado:** 1 taza
* **Sal y pimienta:** al gusto
* **Aceite vegetal:** para freír


**Opcionales:**

* Queso parmesano rallado
* Perejil picado
* Ajo en polvo
--------------------------------------------------------------------------------
CORREGIDO:
## Ingredientes para Milanesas:

* 500g carne de res molida (o cerdo, o pollo)
* 1 huevo
* 1/2 taza pan rallado
* 1/4 taza queso parmesano rallado
* 1 diente ajo picado
* 1/4 taza perejil fresco picado
* Sal y pimienta al gusto
* Aceite vegetal para freír


In [8]:
# ─── Anti-patrón 4: múltiples tareas en un prompt ────────────────────────────
roto = """Explicá qué es el machine learning, también describí sus aplicaciones,
y además dame un ejemplo de código en Python y una lista de libros recomendados."""

# En vez de un prompt gigante, partimos en dos tareas enfocadas
tarea_1 = """Sos un docente universitario. Explicá qué es el machine learning
en 3 oraciones simples, sin código, para alguien sin experiencia técnica."""

tarea_2 = """Sos un docente universitario. Listá 3 aplicaciones reales del machine learning
en empresas. Una línea por aplicación."""

print("ROTO — todo junto:")
print(llamar_llm(roto, max_tokens=200))
print("----" * 20)
print("CORREGIDO — tarea 1:")
print(llamar_llm(tarea_1, max_tokens=120))
print("----" * 20)
print("CORREGIDO — tarea 2:")
print(llamar_llm(tarea_2, max_tokens=120))

ROTO — todo junto:
## Machine Learning: Aprendizaje Automático

El **machine learning (ML)** es una subdisciplina de la inteligencia artificial que permite a las máquinas aprender de los datos sin ser programadas explícitamente. 

En esencia, se trata de entrenar a algoritmos con conjuntos de datos para que puedan identificar patrones, hacer predicciones o tomar decisiones basadas en nuevas entradas.

### Aplicaciones del Machine Learning:

* **Reconocimiento de imágenes y voz:** Identificar objetos, rostros, emociones en imágenes o transcribir audio.
* **Recomendaciones personalizadas:** Sugerir productos, películas o música según el historial de uso del usuario.
* **Predicción de tendencias:** Analizar datos históricos para predecir ventas, precios de acciones o comportamiento futuro.
* **Detección de fraudes:** Identificar transacciones sospechosas en tiempo real.
* **Diagnóstico médico:** Ayudar a los médicos a diagnosticar enfermedades a partir de imágenes médicas y otros datos.



In [9]:
# ─── Anti-patrón 5: negaciones sin alternativa ───────────────────────────────
roto      = "Explicá inteligencia artificial. No uses tecnicismos y no seas aburrido."
corregido = """Explicá inteligencia artificial usando una analogía cotidiana.
Tono: conversacional y entusiasta. Máximo 3 oraciones."""

print("ROTO:")
print(llamar_llm(roto, max_tokens=120))
print("----" * 20)
print("CORREGIDO:")
print(llamar_llm(corregido, max_tokens=120))

ROTO:
Imagina una computadora que aprende como nosotros, pero mucho más rápido. 

La inteligencia artificial (IA) es como eso: programas que pueden aprender de la información que les damos y usarla para hacer cosas inteligentes, como entender nuestro lenguaje, reconocer imágenes o incluso escribir historias. 

Es como si le enseñáramos a un perro a traer el periódico, pero en vez de palabras, le mostramos miles de ejemplos y el programa aprende a hacerlo solo. ¡Y lo mejor es que la IA está mejorando cada día, haciendo cosas increíbles! 😊
--------------------------------------------------------------------------------
CORREGIDO:
¡Imagina la inteligencia artificial como un superaprendiz!  Se alimenta de montañas de información, como libros, fotos e incluso conversaciones, para aprender patrones y hacer predicciones increíbles. ¡Es como si tu cerebro se expandiera exponencialmente y pudiera resolver problemas complejos en un abrir y cerrar de ojos!


---
---
## 5. Actividad: corregir prompts rotos

Cada celda tiene un prompt con un anti-patrón identificado. Tu tarea es reescribir el prompt, ejecutarlo y observar la diferencia.

#### Ejercicio A — Anti-patrón: sin rol ni formato 

In [10]:
# ─── Ejercicio A — Anti-patrón: sin rol ni formato ───────────────────────────
prompt_roto_a = "Decime cómo mejorar mis presentaciones."

print("RESPUESTA CON PROMPT ROTO:")
print(llamar_llm(prompt_roto_a, max_tokens=150))
print()

RESPUESTA CON PROMPT ROTO:
Para mejorar tus presentaciones, sigue estos consejos:

* **Claridad:** Define un objetivo claro y estructura tu presentación de manera lógica. Usa lenguaje sencillo y evita tecnicismos innecesarios.
* **Concisión:** Sé breve y directo. Evita información irrelevante y párrafos largos. 
* **Visuales:** Utiliza imágenes, gráficos y diapositivas atractivas para ilustrar tus puntos. No abuses del texto en las diapositivas.
* **Entusiasmo:** Habla con pasión y energía. Muestra tu interés por el tema y conecta con la audiencia.
* **Práctica:** Repasa tu presentación varias veces para sentirte seguro y natural al momento de hablar.

**Recuerda**: La práctica



In [11]:
# Ejercicio A: Reescribí el prompt con rol, instrucción clara y formato definido
mi_correccion_a = """Sos un experto en comunicación y presentaciones. Dime 5 consejos prácticos para mejorar mis presentaciones, en formato de lista numerada, con máximo 2 líneas por consejo."""

print("RESPUESTA CON TU CORRECCIÓN:")
print(llamar_llm(mi_correccion_a, max_tokens=150))

RESPUESTA CON TU CORRECCIÓN:
Aquí tienes 5 consejos para mejorar tus presentaciones:

1. **Conoce a tu audiencia:** Adapta tu lenguaje y ejemplos a sus intereses e conocimientos previos.
2. **Estructura clara:**  Divide tu presentación en secciones con títulos concisos que guíen al público.
3. **Menos es más:** Evita el exceso de texto en diapositivas, utiliza imágenes impactantes y gráficos claros. 
4. **Practica la entrega:** Repasa tu discurso varias veces para fluir naturalmente y conectar con la audiencia.
5. **Interacción:** Plantea preguntas, realiza encuestas o anima a la participación para mantener la atención.


---
#### Ejercicio B — Anti-patrón: negación sin alternativa

In [12]:
# ─── Ejercicio B — Anti-patrón: negación sin alternativa ─────────────────────
prompt_roto_b = "Escribí un email de ventas. No lo hagas genérico, no uses frases cliché."

print("RESPUESTA CON PROMPT ROTO:")
print(llamar_llm(prompt_roto_b, max_tokens=150))
print()

RESPUESTA CON PROMPT ROTO:
Asunto: [Nombre del cliente] -  Resuelve [Problema específico que tu producto/servicio resuelve].

Hola [Nombre del cliente],

Me llamo [Tu nombre] de [Nombre de tu empresa]. 

Vi en [Plataforma donde encontraste al cliente] que buscas [Menciona algo específico sobre el problema o necesidad del cliente]. 

En [Nombre de tu empresa] ayudamos a empresas como la tuya a [Beneficios concretos que ofrece tu producto/servicio].  [Ejemplo: "Reducir costos en un X%", "Aumentar conversiones en un Y%"]  y lo hacemos a través de [Breve descripción de cómo funciona tu producto/servicio].

¿Te gustaría saber más sobre cómo podemos



In [13]:
# Ejercicio B: Reescribí el prompt describiendo lo que SÍ querés
mi_correccion_b = """Sos un vendedor con 10 años de experiencia en el rubro de serevicios financieros. Escribí un email de ventas para ofrecer nuestro nuevo producto de asesoramiento en inversiones, destacando su funcionalidad y beneficios."""

print("RESPUESTA CON TU CORRECCIÓN:")
print(llamar_llm(mi_correccion_b, max_tokens=150))

RESPUESTA CON TU CORRECCIÓN:
## Asunto: ¡Logra tus metas financieras con [Nombre del servicio]!

Estimado/a [Nombre del cliente],

Como vendedor de servicios financieros con más de 10 años de experiencia, sé lo complejo que puede ser navegar el mundo de las inversiones. Es por eso que te quiero presentar **[Nombre del servicio]**, nuestra nueva plataforma de asesoramiento en inversiones diseñada para ayudarte a alcanzar tus objetivos financieros.

**¿Qué hace que [Nombre del servicio] sea diferente?**

* **Personalización:**  Ofrecemos un análisis personalizado de tu perfil financiero y tolerancia al riesgo para crear una estrategia de inversión única para ti.
* **Diversificación inteligente:** Te ayudamos a diversificar tu portafolio con diferentes


---
#### Ejercicio C: Separar en dos prompts enfocados y ejecutar ambos

In [14]:
# ─── Ejercicio C — Anti-patrón: múltiples tareas ─────────────────────────────
prompt_roto_c = """Explicá qué es una base de datos relacional, también comparala con
NoSQL, y dame cuándo usar cada una y un ejemplo de empresa real para cada caso."""

print("RESPUESTA CON PROMPT ROTO:")
print(llamar_llm(prompt_roto_c, max_tokens=200))
print()

RESPUESTA CON PROMPT ROTO:
## Bases de Datos: Relacionales vs NoSQL

**Bases de Datos Relacionales:** Almacena datos en tablas con filas y columnas, relacionadas entre sí a través de claves comunes. Permiten consultas precisas y transacciones atomizadas (todo o nada). Ideal para aplicaciones que requieren integridad de datos y relaciones complejas.

**Ejemplo:** **Bancos**, como Banco Santander, usan bases de datos relacionales (como MySQL o PostgreSQL) para gestionar cuentas, transacciones y clientes, garantizando la precisión y seguridad de la información financiera.

**Bases de Datos NoSQL:** Más flexibles y escalables que las relacionales, ya que pueden almacenar diferentes tipos de datos (texto, imágenes, documentos) en estructuras no tradicionales como árboles, grafos o key-value.

**Ejemplos:**

* **MongoDB**, usado por **Netflix** para almacenar información sobre usuarios, películas y visualizaciones, adaptándose a la gran cantidad de datos y diversidad de contenido.
* **Cassan

In [15]:
# Ejercicio C: Separar en dos prompts enfocados y ejecutar ambos
mi_prompt_c1 = """Eres un experto en bases de datos. Explicá qué es una base de datos relacional, en 3 oraciones simples, para personal administrativo y da un ejemplo de empresa real que use este tipo de base de datos."""
mi_prompt_c2 = """Eres un experto en bases de datos. Explicá qué es una base de datos NoSQL, en 3 oraciones simples, para personal administrativo y da un ejemplo de empresa real que use este tipo de base de datos."""

print("TAREA 1:")
print(llamar_llm(mi_prompt_c1, max_tokens=130))
print()
print("TAREA 2:")
print(llamar_llm(mi_prompt_c2, max_tokens=130))

TAREA 1:
Una base de datos relacional organiza la información en tablas conectadas por relaciones. Cada tabla tiene filas (registros) y columnas (atributos), permitiendo almacenar y recuperar datos de forma eficiente y estructurada. 

Por ejemplo, Amazon utiliza una base de datos relacional para gestionar información sobre productos, clientes y pedidos, conectando cada elemento a través de relaciones definidas.

TAREA 2:
Una base de datos NoSQL es diferente a las tradicionales porque no usa tablas rígidas con filas y columnas. Permite almacenar información de diversas formas, como documentos, gráficos o claves-valores, adaptándose mejor a grandes volúmenes de datos no estructurados.  Netflix utiliza una base de datos NoSQL para gestionar recomendaciones personalizadas y perfiles de usuario.


---
#### Ejercicios D y E: Diagnóstico y corrección de tus propios prompts

In [ ]:
# ─── Ejercicio D y E — Dos prompts a tu elección ─────────────────────────────
# TODO: Escribí dos prompts rotos propios (de tu trabajo o de un caso que conozcas)
# Identificá el anti-patrón y corregílos.

prompt_roto_d = """..."""
# Anti-patrón detectado: ...
prompt_corregido_d = """..."""

prompt_roto_e = """..."""
# Anti-patrón detectado: ...
prompt_corregido_e = """..."""

for nombre, roto, corregido in [
    ("D", prompt_roto_d, prompt_corregido_d),
    ("E", prompt_roto_e, prompt_corregido_e),
]:
    print(f"=== Ejercicio {nombre} ===")
    print("Roto:")
    print(llamar_llm(roto, max_tokens=130))
    print("Corregido:")
    print(llamar_llm(corregido, max_tokens=130))
    print()